# Base_Metal_download_v3
KOMIS 자원 가격 수집 → investar 적재 (엑셀 저장은 선택)

**저장 테이블 (investar, 표준 규격 STD-1)**

| 테이블 | 내용 | 키 |
|---|---|---|
| `kr_komis_mineral_price_info` | 가격 시리즈 설명표 (광종, 분류, 가격기준, 주기, 단위) | series_id |
| `kr_komis_mineral_price` | 가격 최신본 (시리즈 x 날짜 1행) | series_id, period_end |
| `kr_komis_mineral_price_vintage` | 값이 처음 들어오거나 바뀐 이력 | id |

- `series_id` = `광종코드_가격기준코드` (예: `MNRL0008_501` = 동, 가격기준 501). 한 광종에 가격기준이 여러 개여도 섞이지 않습니다.
- STD-1: 소문자 snake_case / utf8mb4_general_ci / `period_end` + `freq`(D,W,M) / 수집 시각 KST / 이력은 `_vintage` / 단위·통화는 설명표에 둔다.

**사용 순서**: 1~4번 셀 실행 → 5번 수집·적재 (처음은 `FULL_LOAD = True`) → 6번 검증

## 1. 설정 및 접속

In [1]:
import os
import sys
import json
import time
from pathlib import Path
from datetime import date, datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
from sqlalchemy import text


def setup_universal_paths():
    """현재 폴더에서 상위로 올라가며 DATA 폴더를 찾아 sys.path에 추가."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            for p in (str(parent), str(data_folder)):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print("프로젝트 루트:", parent)
            return parent
    raise FileNotFoundError("DATA 폴더를 찾지 못했습니다. investment_strategy 아래에서 실행하세요.")


ROOT = setup_universal_paths()
import config

engine = config.get_engine(config.get_db_info())
KST = timezone(timedelta(hours=9))

T_INFO = "kr_komis_mineral_price_info"
T_DATA = "kr_komis_mineral_price"
T_VIN = "kr_komis_mineral_price_vintage"

# 엑셀 저장 경로 (SAVE_EXCEL = True일 때만 사용)
USER_HOME = Path(r"C:\Users\82108")
SUBPATH = Path("OneDrive") / "INVESTMENT" / "지표상회_복제" / "result_files"


def default_output_dir():
    """1) 환경변수 KOMIS_OUTPUT_DIR  2) USER_HOME/SUBPATH  3) 현재 사용자 홈/SUBPATH"""
    env = os.environ.get("KOMIS_OUTPUT_DIR")
    if env:
        return Path(env)
    if USER_HOME.is_dir():
        return USER_HOME / SUBPATH
    fallback = Path.home() / SUBPATH
    print(f"[안내] {USER_HOME} 를 찾을 수 없습니다. {Path.home()} 기준으로 저장합니다.")
    return fallback

print("DB 엔진 준비 완료")

프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
DB 엔진 준비 완료


## 2. KOMIS 수집기

In [2]:
AVG_TO_FREQ = {"DAY": "D", "WEEK": "W", "MONTH": "M"}


class KOMISCrawler:
    def __init__(self):
        self.base_url = "https://www.komis.or.kr"
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "application/json, text/javascript, */*; q=0.01",
            "Accept-Language": "ko-KR,ko;q=0.9",
            "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
            "X-Requested-With": "XMLHttpRequest",
            "Origin": "https://www.komis.or.kr",
        })

        # avg: 카테고리 기본 평균옵션. DAY 데이터가 없으면 WEEK → MONTH 순으로 자동 재시도
        self.categories = {
            "HP001": {"name": "비철금속", "url": "/Komis/RsrcPrice/BaseMetals", "avg": "DAY",
                      "minerals": {"니켈": "MNRL0002", "동": "MNRL0008", "아연": "MNRL0023",
                                   "알루미늄": "MNRL0009", "연": "MNRL0022", "주석": "MNRL0016"}},
            "HP002": {"name": "희소금속", "url": "/Komis/RsrcPrice/MinorMetals", "avg": "DAY",
                      "minerals": {"리튬": "MNRL0001", "코발트": "MNRL0003", "망간": "MNRL0004",
                                   "크롬": "MNRL0021", "몰리브덴": "MNRL0012", "바나듐": "MNRL0013",
                                   "텅스텐": "MNRL0018", "티타늄": "MNRL0017", "안티모니": "MNRL0019",
                                   "니오븀": "MNRL0007", "규소": "MNRL0010", "마그네슘": "MNRL0011",
                                   "흑연": "MNRL0005", "갈륨": "MNRL0024", "게르마늄": "MNRL0035",
                                   "인듐": "MNRL0025", "셀레늄": "MNRL0029", "탄탈륨": "MNRL0026",
                                   "지르코늄": "MNRL0027", "스트론튬": "MNRL0028"}},
            "HP003": {"name": "철광석및에너지", "url": "/Komis/RsrcPrice/IronOre", "avg": "WEEK",
                      "minerals": {"유연탄": "MNRL0032"}},   # 철광석 코드는 웹사이트 확인 필요
            "HP004": {"name": "기타광석", "url": "/Komis/RsrcPrice/EtcMnrl", "avg": "DAY",
                      "minerals": {"금": "MNRL0046", "은": "MNRL0047", "백금": "MNRL0014",
                                   "팔라듐": "MNRL0015", "우라늄": "MNRL0031"}},
        }

    def update_referer(self, category_url):
        self.session.headers["Referer"] = self.base_url + category_url

    def get_price_criteria(self, mineral_code, hp000="HP001"):
        """광종별 가격기준 조회. 반환: (코드, 이름, 원본 dict)"""
        url = f"{self.base_url}/Komis/RsrcPrice/ajax/getMnrlPriceCrtr"
        try:
            r = self.session.post(url, data={"HP000": hp000, "mnrkndUnqCd": mineral_code}, timeout=10)
            if r.status_code == 200:
                rows = r.json().get("data") or []
                if rows:
                    item = rows[0]
                    name = None
                    for k in ("cdNm", "cdKeyNm", "cdValue", "cdVal", "codeNm", "prcCrtrNm"):
                        if item.get(k):
                            name = str(item[k])
                            break
                    return str(item.get("cdKey", "") or ""), name, item
        except Exception as e:
            print(f"  가격기준 조회 오류: {e}")
        return "", None, {}

    def get_price_data(self, mineral_code, price_criteria, start_year, end_year, avg_option, hp000):
        """가격 데이터 원본 JSON 조회. 실패 시 None."""
        url = f"{self.base_url}/Komis/RsrcPrice/ajax/getMnrlPrcByMnrkndUnqCd"
        params = {
            "HP000": hp000, "srchMnrkndUnqCd": mineral_code, "srchPrcCrtr": price_criteria,
            "srchAvgOpt": avg_option, "srchField": "year",
            "srchStartDate": str(start_year), "srchEndDate": str(end_year),
            "srchCompareMnrkndUnqCd": "", "srchComparePrcCrtr": "", "lmeInvt": "N",
        }
        try:
            r = self.session.post(url, data=params, timeout=30)
            if r.status_code == 200:
                return r.json()
            print(f"  데이터 조회 실패: HTTP {r.status_code}")
        except Exception as e:
            print(f"  데이터 조회 오류: {e}")
        return None

    def fetch_series(self, mineral_name, hp000, start_year, end_year, avg_option=None, chunk_years=3):
        """한 광종의 가격을 표준 형태 DataFrame으로 반환.
        반환: (df[crtrYmd, price, change, change_pct], meta dict) 또는 (None, meta)
        """
        cat = self.categories[hp000]
        mineral_code = cat["minerals"][mineral_name]
        self.update_referer(cat["url"])
        crit_code, crit_name, crit_raw = self.get_price_criteria(mineral_code, hp000)

        meta = {"mineral_code": mineral_code, "mineral_name": mineral_name,
                "category_code": hp000, "category_name": cat["name"],
                "price_criteria_code": crit_code, "price_criteria_name": crit_name,
                "criteria_raw": crit_raw, "info_raw": {}, "avg_option": None}

        tries = [avg_option or cat["avg"]]
        for alt in ("WEEK", "MONTH"):
            if alt not in tries:
                tries.append(alt)

        for avg in tries:
            frames = []
            for y0 in range(start_year, end_year + 1, chunk_years):
                y1 = min(y0 + chunk_years - 1, end_year)
                res = self.get_price_data(mineral_code, crit_code, y0, y1, avg, hp000)
                time.sleep(0.5)
                if not res:
                    continue
                info = (res.get("dataAvg") or {}).get("INFO") or {}
                if info:
                    meta["info_raw"] = info
                rows = ((res.get("data") or {}).get("defaultMnrl")) or []
                if rows:
                    frames.append(pd.DataFrame(rows))
            if frames:
                df = pd.concat(frames, ignore_index=True)
                need = ["crtrYmd", "cmercPrc", "flctnPrc", "flctnPrcnt"]
                for c in need:
                    if c not in df.columns:
                        df[c] = np.nan
                df = df[need].rename(columns={"cmercPrc": "price", "flctnPrc": "change",
                                              "flctnPrcnt": "change_pct"})
                for c in ("price", "change", "change_pct"):
                    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False)
                                          .str.replace("%", "", regex=False), errors="coerce")
                df["crtrYmd"] = df["crtrYmd"].astype(str).str.strip()
                df = df.drop_duplicates("crtrYmd", keep="last")
                meta["avg_option"] = avg
                return df, meta
            print(f"  {mineral_name}: {avg} 데이터 없음")
        return None, meta


def save_excel(df, meta, start_year, end_year, output_dir=None):
    """기존 v2와 같은 형식으로 엑셀 저장 (선택 기능)."""
    output_dir = Path(output_dir or default_output_dir())
    output_dir.mkdir(parents=True, exist_ok=True)
    out = df.rename(columns={"crtrYmd": "기준일", "price": "기준가격",
                             "change": "등락가", "change_pct": "등락율(%)"}).copy()
    if meta["avg_option"] in ("DAY", "WEEK"):
        out["기준일"] = pd.to_datetime(out["기준일"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
    else:
        out["기준일"] = out["기준일"].str[:4] + "-" + out["기준일"].str[4:6]
    fn = (f"{meta['category_name']}_{meta['mineral_name']}_{meta['avg_option']}_"
          f"{start_year}_{end_year}.xlsx")
    out.to_excel(output_dir / fn, index=False, engine="openpyxl")
    return output_dir / fn

print("수집기 준비 완료")

수집기 준비 완료


## 3. 테이블 생성 (없을 때만)

In [3]:
_OPTS = "ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci"

DDL = [
f"""
CREATE TABLE IF NOT EXISTS {T_INFO} (
    series_id           VARCHAR(50)  NOT NULL COMMENT '광종코드_가격기준코드',
    mineral_code        VARCHAR(20)  NOT NULL COMMENT 'KOMIS 광종코드 (MNRL....)',
    mineral_name        VARCHAR(50)  NULL     COMMENT '광종명(한글)',
    category_code       VARCHAR(10)  NULL     COMMENT 'HP001~HP004',
    category_name       VARCHAR(50)  NULL     COMMENT '비철금속, 희소금속 등',
    price_criteria_code VARCHAR(20)  NULL     COMMENT 'KOMIS 가격기준 코드',
    price_criteria_name VARCHAR(100) NULL,
    freq                CHAR(1)      NOT NULL COMMENT 'D,W,M',
    unit                VARCHAR(50)  NULL,
    currency            CHAR(3)      NULL     COMMENT 'ISO 4217',
    criteria_raw        TEXT         NULL     COMMENT '가격기준 응답 원본(JSON)',
    info_raw            TEXT         NULL     COMMENT '광종 정보 응답 원본(JSON)',
    is_active           TINYINT      NOT NULL DEFAULT 1,
    first_collected_at  DATETIME     NOT NULL,
    last_collected_at   DATETIME     NOT NULL,
    created_at          TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at          TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (series_id),
    KEY idx_mineral (mineral_code),
    KEY idx_category (category_code)
) {_OPTS} COMMENT='KOMIS 자원 가격 시리즈 설명표 (STD-1)'
""",
f"""
CREATE TABLE IF NOT EXISTS {T_DATA} (
    series_id          VARCHAR(50) NOT NULL,
    period_end         DATE        NOT NULL COMMENT '기준일 (월 데이터는 월말일)',
    freq               CHAR(1)     NOT NULL,
    price              DOUBLE      NULL     COMMENT '기준가격',
    change_amt         DOUBLE      NULL     COMMENT '등락가',
    change_pct         DOUBLE      NULL     COMMENT '등락율(%)',
    first_collected_at DATETIME    NOT NULL COMMENT '최초 관측 시각(KST)',
    last_collected_at  DATETIME    NOT NULL COMMENT '최근 관측 시각(KST)',
    created_at         TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at         TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (series_id, period_end),
    KEY idx_period_end (period_end)
) {_OPTS} COMMENT='KOMIS 자원 가격 최신본 (STD-1)'
""",
f"""
CREATE TABLE IF NOT EXISTS {T_VIN} (
    id           BIGINT      NOT NULL AUTO_INCREMENT,
    series_id    VARCHAR(50) NOT NULL,
    period_end   DATE        NOT NULL,
    price        DOUBLE      NULL,
    change_type  VARCHAR(10) NOT NULL COMMENT 'new | revised',
    collected_at DATETIME    NOT NULL COMMENT '이 값을 관측한 시각(KST)',
    created_at   TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (id),
    KEY idx_key_time (series_id, period_end, collected_at)
) {_OPTS} COMMENT='KOMIS 자원 가격 관측 이력 (STD-1 vintage)'
""",
]

with engine.begin() as cx:
    for ddl in DDL:
        cx.execute(text(ddl))
print("테이블 준비 완료:", T_INFO, T_DATA, T_VIN)

테이블 준비 완료: kr_komis_mineral_price_info kr_komis_mineral_price kr_komis_mineral_price_vintage


## 4. 적재 함수

In [4]:
def now_kst():
    return datetime.now(KST).replace(tzinfo=None, microsecond=0)


def to_period_end(ymd, freq):
    """KOMIS 기준일(YYYYMMDD 또는 YYYYMM) → 기간 말일 date."""
    s = str(ymd).strip()
    if freq == "M" or len(s) == 6:
        return (pd.Timestamp(year=int(s[:4]), month=int(s[4:6]), day=1) + pd.offsets.MonthEnd(0)).date()
    return datetime.strptime(s[:8], "%Y%m%d").date()


def _find_unit(meta):
    """응답 원본에서 단위 후보 키를 찾아 반환 (없으면 None)."""
    for src in (meta.get("info_raw") or {}, meta.get("criteria_raw") or {}):
        for k, v in src.items():
            if v and any(t in k.lower() for t in ("unit", "untnm", "unt")):
                return str(v)[:50]
    return None


def _currency(unit):
    if not unit:
        return None
    u = unit.upper()
    if "$" in u or "USD" in u or "달러" in unit:
        return "USD"
    if "원" in unit or "KRW" in u:
        return "KRW"
    if "CNY" in u or "RMB" in u or "위안" in unit:
        return "CNY"
    return None


def _same(a, b):
    if a is None or b is None:
        return a is None and b is None
    return abs(float(a) - float(b)) <= 1e-9 * max(1.0, abs(float(a)))


def make_series_id(meta):
    crit = meta.get("price_criteria_code") or ""
    return f"{meta['mineral_code']}_{crit}" if crit else meta["mineral_code"]


def upsert_info(meta, freq, ts):
    unit = _find_unit(meta)
    rec = {
        "series_id": make_series_id(meta), "mineral_code": meta["mineral_code"],
        "mineral_name": meta["mineral_name"], "category_code": meta["category_code"],
        "category_name": meta["category_name"],
        "price_criteria_code": meta.get("price_criteria_code") or None,
        "price_criteria_name": meta.get("price_criteria_name"),
        "freq": freq, "unit": unit, "currency": _currency(unit),
        "criteria_raw": json.dumps(meta.get("criteria_raw") or {}, ensure_ascii=False),
        "info_raw": json.dumps(meta.get("info_raw") or {}, ensure_ascii=False),
        "ts": ts,
    }
    sql = f"""
    INSERT INTO {T_INFO}
        (series_id, mineral_code, mineral_name, category_code, category_name,
         price_criteria_code, price_criteria_name, freq, unit, currency,
         criteria_raw, info_raw, first_collected_at, last_collected_at)
    VALUES
        (:series_id, :mineral_code, :mineral_name, :category_code, :category_name,
         :price_criteria_code, :price_criteria_name, :freq, :unit, :currency,
         :criteria_raw, :info_raw, :ts, :ts)
    ON DUPLICATE KEY UPDATE
        mineral_name = VALUES(mineral_name), category_code = VALUES(category_code),
        category_name = VALUES(category_name), price_criteria_name = VALUES(price_criteria_name),
        freq = VALUES(freq), unit = COALESCE(VALUES(unit), unit),
        currency = COALESCE(VALUES(currency), currency),
        criteria_raw = VALUES(criteria_raw), info_raw = VALUES(info_raw),
        last_collected_at = VALUES(last_collected_at)
    """
    with engine.begin() as cx:
        cx.execute(text(sql), rec)
    return rec["series_id"]


def upsert_prices(series_id, df, freq, ts):
    """가격 저장. 신규 날짜나 값이 바뀐 날짜는 vintage에도 기록. 반환: (저장 건수, 이력 건수)"""
    recs = {}
    for ymd, p, c, cp in zip(df["crtrYmd"], df["price"], df["change"], df["change_pct"]):
        if pd.isna(p):
            continue
        pe = to_period_end(ymd, freq)
        recs[pe] = {"series_id": series_id, "period_end": pe, "freq": freq, "price": float(p),
                    "change_amt": None if pd.isna(c) else float(c),
                    "change_pct": None if pd.isna(cp) else float(cp), "ts": ts}
    recs = list(recs.values())
    if not recs:
        return 0, 0
    upsert = f"""
    INSERT INTO {T_DATA}
        (series_id, period_end, freq, price, change_amt, change_pct, first_collected_at, last_collected_at)
    VALUES
        (:series_id, :period_end, :freq, :price, :change_amt, :change_pct, :ts, :ts)
    ON DUPLICATE KEY UPDATE
        freq = VALUES(freq), price = VALUES(price),
        change_amt = VALUES(change_amt), change_pct = VALUES(change_pct),
        first_collected_at = LEAST(first_collected_at, VALUES(first_collected_at)),
        last_collected_at  = GREATEST(last_collected_at, VALUES(last_collected_at))
    """
    ins_vin = f"""
    INSERT INTO {T_VIN} (series_id, period_end, price, change_type, collected_at)
    VALUES (:series_id, :period_end, :price, :change_type, :ts)
    """
    with engine.begin() as cx:
        existing = {pe: v for pe, v in cx.execute(
            text(f"SELECT period_end, price FROM {T_DATA} WHERE series_id = :s"), {"s": series_id})}
        vin = []
        for r in recs:
            if r["period_end"] not in existing:
                vin.append(dict(r, change_type="new"))
            elif not _same(existing[r["period_end"]], r["price"]):
                vin.append(dict(r, change_type="revised"))
        cx.execute(text(upsert), recs)
        if vin:
            cx.execute(text(ins_vin), vin)
    return len(recs), len(vin)


def collect_to_db(start_year, end_year, categories=None, minerals=None,
                  save_to_excel=False, output_dir=None):
    """KOMIS 가격을 받아 investar에 적재 (원하면 엑셀도 저장)."""
    crawler = KOMISCrawler()
    ts = now_kst()
    summary = []
    for hp000 in (categories or list(crawler.categories)):
        cat = crawler.categories[hp000]
        print(f"\n===== {cat['name']} ({hp000}) {start_year}~{end_year} =====")
        for name in cat["minerals"]:
            if minerals and name not in minerals:
                continue
            try:
                df, meta = crawler.fetch_series(name, hp000, start_year, end_year)
                if df is None or df.empty:
                    print(f"[FAIL] {name}: 데이터 없음")
                    summary.append((cat["name"], name, None, 0, 0, "데이터 없음"))
                    continue
                freq = AVG_TO_FREQ[meta["avg_option"]]
                sid = upsert_info(meta, freq, ts)
                n, nv = upsert_prices(sid, df, freq, ts)
                if save_to_excel:
                    save_excel(df, meta, start_year, end_year, output_dir)
                print(f"[OK]   {name:<6} {sid:<14} freq={freq}  저장 {n:>5}건, 신규/수정 {nv:>5}건")
                summary.append((cat["name"], name, sid, n, nv, "OK"))
            except Exception as e:
                print(f"[FAIL] {name}: {type(e).__name__}: {e}")
                summary.append((cat["name"], name, None, 0, 0, f"{type(e).__name__}"))
            time.sleep(1)
        time.sleep(2)
    res = pd.DataFrame(summary, columns=["category", "mineral", "series_id", "n_saved", "n_new_or_revised", "status"])
    print(f"\n성공 {int((res.status == 'OK').sum())}/{len(res)}")
    return res


def load_komis_wide(category_code=None, start=None, value="price"):
    """DB에서 읽어 광종명을 컬럼으로 하는 표로 반환 (index = period_end)."""
    q = (f"SELECT d.period_end, i.mineral_name, d.{value} AS v FROM {T_DATA} d "
         f"JOIN {T_INFO} i ON d.series_id = i.series_id WHERE 1 = 1")
    params = {}
    if category_code:
        q += " AND i.category_code = :c"
        params["c"] = category_code
    if start:
        q += " AND d.period_end >= :st"
        params["st"] = start
    with engine.connect() as cx:
        long = pd.read_sql(text(q), cx, params=params)
    if long.empty:
        return long
    long["period_end"] = pd.to_datetime(long["period_end"])
    return long.pivot_table(index="period_end", columns="mineral_name", values="v", aggfunc="last").sort_index()

print("적재 함수 준비 완료")

적재 함수 준비 완료


## 5. 수집 및 적재
- 처음 한 번: `FULL_LOAD = True` → `BACKFILL_START_YEAR`부터 전체 적재
- 이후 정기 실행: `FULL_LOAD = False` → 작년~올해만 받아 갱신

In [5]:
FULL_LOAD = False
BACKFILL_START_YEAR = 2010     # KOMIS 제공 범위보다 이르면 해당 구간은 빈 응답으로 넘어감
SAVE_EXCEL = False             # True면 기존처럼 엑셀도 저장

this_year = date.today().year
start_year = BACKFILL_START_YEAR if FULL_LOAD else this_year - 1

result = collect_to_db(start_year, this_year, save_to_excel=SAVE_EXCEL)
result


===== 비철금속 (HP001) 2025~2026 =====
[OK]   니켈     MNRL0002_502   freq=D  저장   442건, 신규/수정   442건
[OK]   동      MNRL0008_501   freq=D  저장   442건, 신규/수정   442건
[OK]   아연     MNRL0023_561   freq=D  저장   442건, 신규/수정   442건
[OK]   알루미늄   MNRL0009_495   freq=D  저장   442건, 신규/수정   442건
[OK]   연      MNRL0022_499   freq=D  저장   442건, 신규/수정   442건
[OK]   주석     MNRL0016_493   freq=D  저장   442건, 신규/수정   442건

===== 희소금속 (HP002) 2025~2026 =====
[OK]   리튬     MNRL0001_773   freq=D  저장   411건, 신규/수정   411건
[OK]   코발트    MNRL0003_791   freq=D  저장   400건, 신규/수정   400건
[OK]   망간     MNRL0004_815   freq=D  저장   410건, 신규/수정   410건
[OK]   크롬     MNRL0021_794   freq=D  저장   411건, 신규/수정   411건
[OK]   몰리브덴   MNRL0012_763   freq=D  저장   410건, 신규/수정   410건
[OK]   바나듐    MNRL0013_775   freq=D  저장   410건, 신규/수정   410건
[OK]   텅스텐    MNRL0018_796   freq=D  저장   413건, 신규/수정   413건
[OK]   티타늄    MNRL0017_761   freq=D  저장   414건, 신규/수정   414건
[OK]   안티모니   MNRL0019_780   freq=D  저장   413건, 신규/수정   413건
[OK]   니오븀   

,category,mineral,series_id,n_saved,n_new_or_revised,status
0,비철금속,니켈,MNRL0002_502,442,442,OK
1,비철금속,동,MNRL0008_501,442,442,OK
2,비철금속,아연,MNRL0023_561,442,442,OK
3,비철금속,알루미늄,MNRL0009_495,442,442,OK
4,비철금속,연,MNRL0022_499,442,442,OK
5,비철금속,주석,MNRL0016_493,442,442,OK
6,희소금속,리튬,MNRL0001_773,411,411,OK
7,희소금속,코발트,MNRL0003_791,400,400,OK
8,희소금속,망간,MNRL0004_815,410,410,OK
9,희소금속,크롬,MNRL0021_794,411,411,OK


## 6. 적재 결과 검증

In [8]:
check_sql = f"""
SELECT i.category_name, i.mineral_name, i.series_id, i.price_criteria_name, i.freq, i.unit,
       COUNT(d.period_end)      AS n_obs,
       MIN(d.period_end)        AS first_date,
       MAX(d.period_end)        AS last_date,
       MAX(d.last_collected_at) AS last_collected_at
FROM {T_INFO} i
LEFT JOIN {T_DATA} d ON d.series_id = i.series_id
GROUP BY i.category_name, i.mineral_name, i.series_id, i.price_criteria_name, i.freq, i.unit
ORDER BY i.category_code, i.mineral_code
"""
with engine.connect() as cx:
    check = pd.read_sql(text(check_sql), cx)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", None)
check

,category_name,mineral_name,series_id,price_criteria_name,freq,unit,n_obs,first_date,last_date,last_collected_at
0,비철금속,니켈,MNRL0002_502,LME CASH,D,ton,442,2025-01-02,2026-09-30,2026-10-01 14:59:11
1,비철금속,동,MNRL0008_501,LME CASH,D,ton,442,2025-01-02,2026-09-30,2026-10-01 14:59:11
2,비철금속,알루미늄,MNRL0009_495,LME CASH,D,ton,442,2025-01-02,2026-09-30,2026-10-01 14:59:11
3,비철금속,주석,MNRL0016_493,LME CASH,D,ton,442,2025-01-02,2026-09-30,2026-10-01 14:59:11
4,비철금속,연,MNRL0022_499,LME CASH,D,ton,442,2025-01-02,2026-09-30,2026-10-01 14:59:11
5,비철금속,아연,MNRL0023_561,LME CASH,D,ton,442,2025-01-02,2026-09-30,2026-10-01 14:59:11
6,희소금속,리튬,MNRL0001_773,Lithium Carbonate,D,kg,411,2025-01-02,2026-09-24,2026-10-01 14:59:11
7,희소금속,코발트,MNRL0003_791,Cobalt Metal,D,kg,400,2025-01-02,2026-09-24,2026-10-01 14:59:11
8,희소금속,망간,MNRL0004_815,Ferro-manganese,D,mt,410,2025-01-02,2026-09-24,2026-10-01 14:59:11
9,희소금속,흑연,MNRL0005_801,Flake Graphite,D,mt,409,2025-01-02,2026-09-24,2026-10-01 14:59:11


## 7. 예시: DB에서 비철금속 가격 읽기

In [9]:
base = load_komis_wide(category_code="HP001", start="2024-01-01")
base.head(10)

mineral_name,니켈,동,아연,알루미늄,연,주석
period_end,,,,,,
2025-01-02,15010.0,8685.5,2926.5,2536.0,1915.0,28225.0
2025-01-03,14770.0,8701.0,2864.5,2475.0,1898.0,28625.0
2025-01-06,15070.0,8893.5,2879.0,2480.0,1918.0,29250.0
2025-01-07,15120.0,8886.0,2844.0,2484.0,1907.0,29550.0
2025-01-08,15180.0,8847.0,2798.0,2474.0,1920.0,30050.0
2025-01-09,15185.0,8960.5,2821.0,2502.5,1907.0,29945.0
2025-01-10,15400.0,8995.5,2856.5,2560.5,1949.0,29825.0
2025-01-13,15560.0,8980.0,2839.0,2573.0,1926.0,29605.0
2025-01-14,15600.0,9016.0,2831.0,2554.5,1914.5,29550.0
